# Minimal RAG on Elasticsearch (ELSER sparse + hybrid search)

Third notebook in this learning path:

1. [`minimal-rag-on-elastic-docs.ipynb`](minimal-rag-on-elastic-docs.ipynb) — Chroma + local **dense** vectors (MiniLM)
2. [`minimal-rag-on-elasticsearch.ipynb`](minimal-rag-on-elasticsearch.ipynb) — Elastic **`semantic_text`** (**dense**) + hybrid RRF
3. **This notebook** — Elastic **ELSER** (**sparse** semantic) + hybrid RRF

**Same RAG loop:** chunk (§3) → index (§4) → retrieve (§5) → prompt (§6) → Ollama (§7 optional).

**Prerequisites:**
- Same env vars as the dense Elastic notebook: `ELASTICSEARCH_URL`, `ELASTICSEARCH_API_KEY`
- **ELSER** available on your deployment (see §2 — check before bulk ingest)
- Docs: [Semantic search with ELSER](https://www.elastic.co/docs/solutions/search/semantic-search/semantic-search-elser-ingest-pipelines), [Download and deploy ELSER](https://www.elastic.co/docs/explore-analyze/machine-learning/nlp/ml-nlp-elser.md)

**Note:** ELSER ingest is **slower** than `semantic_text` (each chunk runs through the ML model). Default `MAX_FILES = 50` keeps the first run manageable.

**License:** CC BY-NC-ND on `docs-content` — personal learning only.

## 1. Install packages

Same venv as the other notebooks is fine:

```bash
cd learning-rag
source .venv/bin/activate
pip install -r requirements-elasticsearch.txt
```

In [1]:
%pip install -q elasticsearch tqdm

You should consider upgrading via the '/Users/mohan/repos/docs-content/learning-rag/.venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Connect to Elastic + check ELSER

Run this cell first. It connects to your cluster and reports whether ELSER looks ready.

### How to confirm ELSER in Kibana (manual check)

1. Open **Kibana** for the same deployment you used in the dense Elastic notebook.
2. Go to **Machine Learning → Trained Models**.
3. Find **ELSER** (model id `.elser_model_2`). Status should be **Started** / deployed (not merely downloaded).
4. Optional: **Stack Management → Inference Endpoints** — look for a `sparse_embedding` endpoint (this notebook creates `elser_rag_embeddings` if missing).

### Requirements on Elastic Cloud Hosted

ELSER needs **ML capacity** (typically a deployment with ML nodes, or autoscaling ML). If §2b fails to create the endpoint, see [Download and deploy ELSER](https://www.elastic.co/docs/explore-analyze/machine-learning/nlp/ml-nlp-elser.md).

**Serverless:** ELSER via ingest pipelines may differ from Hosted; if checks fail, use the dense `semantic_text` notebook instead.

In [11]:
import os
import re
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.exceptions import NotFoundError


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "get-started").is_dir() and (p / "README.md").is_file():
            return p
    return (here / "..").resolve()


def corpus_to_index_name(folder: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", folder.lower()).strip("-")
    return f"elastic-rag-{slug}-elser"[:220]


ELASTICSEARCH_URL = os.environ.get("ELASTICSEARCH_URL", "").strip()
ELASTICSEARCH_API_KEY = os.environ.get("ELASTICSEARCH_API_KEY", "").strip()

if not ELASTICSEARCH_URL or not ELASTICSEARCH_API_KEY:
    raise EnvironmentError(
        "Set ELASTICSEARCH_URL and ELASTICSEARCH_API_KEY in your shell before Jupyter."
    )

REPO_ROOT = find_repo_root()
CORPUS_FOLDER = "solutions"
MAX_FILES = 50  # ELSER ingest is slow; raise after a successful first run

CORPUS_SUBDIR = (REPO_ROOT / CORPUS_FOLDER).resolve()
INDEX_NAME = corpus_to_index_name(CORPUS_FOLDER)
PIPELINE_ID = "elastic-rag-elser-pipeline"
ELSER_INFERENCE_ID = "elser_rag_embeddings"
ELSER_MODEL_ID = ".elser_model_2"

assert CORPUS_SUBDIR.is_dir(), f"Missing folder: {CORPUS_SUBDIR}"

es = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY)
info = es.info()
print("Connected:", info.get("cluster_name", "?"), "| ES", info.get("version", {}).get("number", "?"))
print("Corpus:", CORPUS_SUBDIR)
print("Index:", INDEX_NAME)
print("Pipeline:", PIPELINE_ID)
print("ELSER inference id (for search):", ELSER_INFERENCE_ID)

Connected: a719d742f0c344f9ba1f9be882898202 | ES 9.5.0
Corpus: /Users/mohan/repos/docs-content/solutions
Index: elastic-rag-solutions-elser
Pipeline: elastic-rag-elser-pipeline
ELSER inference id (for search): elser_rag_embeddings


In [13]:
def check_elser_status(es: Elasticsearch) -> None:
    print("\n=== ELSER status ===")

    # 1) Inference endpoint (used at query time for sparse_vector)
    try:
        es.inference.get(inference_id=ELSER_INFERENCE_ID, task_type="sparse_embedding")
        print(f"[OK] Inference endpoint '{ELSER_INFERENCE_ID}' exists")
    except NotFoundError:
        print(f"[--] Inference endpoint '{ELSER_INFERENCE_ID}' not found yet (§2b will create it)")
    except Exception as exc:
        print(f"[??] Could not read inference endpoint: {exc}")

    # 2) Trained model artifact
    try:
        models = es.ml.get_trained_models(model_id=ELSER_MODEL_ID, allow_no_match=True)
        if models.get("count", 0):
            print(f"[OK] Trained model {ELSER_MODEL_ID} is present in the cluster")
        else:
            print(f"[--] Trained model {ELSER_MODEL_ID} not downloaded yet (§2b download may take several minutes)")
    except Exception as exc:
        print(f"[??] Could not query trained models: {exc}")

    # 3) Deployment stats
    try:
        stats = es.ml.get_trained_models_stats(model_id=ELSER_MODEL_ID, allow_no_match=True)
        started = []
        for m in stats.get("trained_model_stats", []):
            ds = m.get("deployment_stats") or {}
            state = ds.get("state", "unknown")
            dep_id = ds.get("deployment_id", "?")
            if ds:
                started.append(f"{dep_id} ({state})")
        if started:
            print(f"[OK] ELSER deployment(s): {', '.join(started)}")
        else:
            print("[--] No deployment stats yet — run §2b and wait for model download/start in Kibana ML")
    except Exception as exc:
        print(f"[??] Could not read deployment stats: {exc}")


check_elser_status(es)


=== ELSER status ===
[OK] Inference endpoint 'elser_rag_embeddings' exists
[OK] Trained model .elser_model_2 is present in the cluster
[OK] ELSER deployment(s): .elser_model_2 (started), elser_rag_embeddings (started)


## 2b. Deploy ELSER, then pipeline + inference endpoint

**Order matters:**
1. **Start** `.elser_model_2` (download + deploy — often **10–20 min** first time)
2. **Ingest pipeline** (for §4 indexing)
3. **Inference endpoint** `elser_rag_embeddings` (for §5 sparse search)

If you see **`model_deployment_timeout_exception` (408)**, the model was not **Started** within the cluster’s short wait. Run this cell again after Kibana shows **Started**, or let step 1 below finish.

**Manual alternative:** Kibana → **ML → Trained Models** → `.elser_model_2` → **Start deployment** → wait until **Started** → re-run this cell.

In [15]:
import time

from elasticsearch import ApiError


# Elastic reports started or fully_allocated when the model is ready for inference
ELSER_READY_STATES = {"started", "fully_allocated"}


def elser_deployment_state(es: Elasticsearch):
    stats = es.ml.get_trained_models_stats(model_id=ELSER_MODEL_ID, allow_no_match=True)
    for m in stats.get("trained_model_stats", []):
        ds = m.get("deployment_stats") or {}
        if ds:
            return ds.get("allocation_status", {}).get("state") or ds.get("state")
    return None


def ensure_elser_deployed(es: Elasticsearch, max_wait_minutes: int = 25) -> None:
    state = elser_deployment_state(es)
    if state in ELSER_READY_STATES:
        print(f"[OK] ELSER deployment ready ({ELSER_MODEL_ID}, state={state})")
        return

    print(
        f"Starting ELSER ({ELSER_MODEL_ID}) — first time often downloads 10–20 min..."
    )
    try:
        es.options(request_timeout=1800).ml.start_trained_model_deployment(
            model_id=ELSER_MODEL_ID,
            wait_for="started",
            timeout="25m",
        )
        print("[OK] start_trained_model_deployment finished")
        return
    except Exception as exc:
        msg = str(exc)
        if "existing deployment" in msg.lower():
            print("Note: deployment already exists (normal on re-run). Checking state...")
        else:
            print(f"Note: start_trained_model_deployment returned: {exc}")
        print(f"Polling until state is one of {ELSER_READY_STATES}...")

    deadline = time.time() + max_wait_minutes * 60
    while time.time() < deadline:
        state = elser_deployment_state(es)
        print(f"  deployment state: {state!r}")
        if state in ELSER_READY_STATES:
            print(f"[OK] ELSER deployment ready (state={state})")
            return
        time.sleep(30)

    raise TimeoutError(
        f"ELSER not started after {max_wait_minutes} minutes. "
        "In Kibana: ML → Trained Models → .elser_model_2 → Start deployment, wait for Started, then re-run §2b."
    )


def ensure_elser_ingest_pipeline(es: Elasticsearch) -> None:
    body = {
        "description": "ELSER sparse embeddings for elastic-rag learning notebook",
        "processors": [
            {
                "inference": {
                    "model_id": ELSER_MODEL_ID,
                    "input_output": [
                        {
                            "input_field": "content",
                            "output_field": "content_embedding",
                        }
                    ],
                }
            }
        ],
    }
    es.ingest.put_pipeline(id=PIPELINE_ID, body=body)
    print("Ingest pipeline ready:", PIPELINE_ID)


def ensure_elser_inference_endpoint(es: Elasticsearch) -> None:
    try:
        es.inference.get(inference_id=ELSER_INFERENCE_ID, task_type="sparse_embedding")
        print("Inference endpoint already exists:", ELSER_INFERENCE_ID)
        return
    except NotFoundError:
        pass

    print("Creating inference endpoint", ELSER_INFERENCE_ID, "...")
    endpoint_body = {
        "service": "elasticsearch",
        "service_settings": {
            "adaptive_allocations": {
                "enabled": True,
                "min_number_of_allocations": 1,
                "max_number_of_allocations": 4,
            },
            "num_threads": 1,
            "model_id": ELSER_MODEL_ID,
        },
    }
    try:
        es.options(request_timeout=600).inference.put(
            inference_id=ELSER_INFERENCE_ID,
            task_type="sparse_embedding",
            body=endpoint_body,
        )
        print("Created inference endpoint:", ELSER_INFERENCE_ID)
    except ApiError as err:
        if err.meta.status == 408:
            print(
                "\n408 = cluster timed out waiting for deployment (often still downloading)."
            )
            print(
                "Wait until Kibana shows .elser_model_2 as Started, then re-run §2b only."
            )
        raise


ensure_elser_deployed(es)
ensure_elser_ingest_pipeline(es)
ensure_elser_inference_endpoint(es)
check_elser_status(es)

[OK] ELSER deployment ready (.elser_model_2, state=fully_allocated)
Ingest pipeline ready: elastic-rag-elser-pipeline
Inference endpoint already exists: elser_rag_embeddings

=== ELSER status ===
[OK] Inference endpoint 'elser_rag_embeddings' exists
[OK] Trained model .elser_model_2 is present in the cluster
[OK] ELSER deployment(s): .elser_model_2 (started), elser_rag_embeddings (started)


## 3. Load Markdown, strip frontmatter, chunk

Identical to the other notebooks. ELSER uses only the **first ~512 tokens** per field — our chunks are sized to stay roughly within that.

In [17]:
def strip_yaml_frontmatter(text: str) -> str:
    text = text.lstrip("\ufeff")
    if not text.startswith("---"):
        return text
    parts = text.split("---", 2)
    if len(parts) >= 3:
        return parts[2].strip()
    return text


def chunk_text(text: str, max_chars: int = 1200, overlap: int = 200):
    text = text.strip()
    chunks = []
    i = 0
    while i < len(text):
        end = min(i + max_chars, len(text))
        if end < len(text):
            para = text.rfind("\n\n", i, end)
            if para != -1 and para > i + max_chars // 2:
                end = para
        piece = text[i:end].strip()
        if len(piece) > 80:
            chunks.append(piece)
        if end >= len(text):
            break
        i = max(i + 1, end - overlap)
    return chunks


def load_corpus(root: Path, max_files: int):
    paths = sorted(root.rglob("*.md"))[:max_files]
    docs = []
    for path in paths:
        try:
            raw = path.read_text(encoding="utf-8", errors="replace")
        except OSError:
            continue
        body = strip_yaml_frontmatter(raw)
        rel = path.relative_to(root)
        for idx, ch in enumerate(chunk_text(body)):
            docs.append(
                {
                    "id": f"{rel.as_posix()}#{idx}",
                    "document": ch,
                    "metadata": {"source": rel.as_posix()},
                }
            )
    return docs


# Run §2 first so CORPUS_SUBDIR and MAX_FILES exist
print("Section 3 starting — loading from:", CORPUS_SUBDIR)
print("MAX_FILES:", MAX_FILES)

documents = load_corpus(CORPUS_SUBDIR, MAX_FILES)
print(f"Chunks ready: {len(documents)} (from up to {MAX_FILES} .md files)")

Section 3 starting — loading from: /Users/mohan/repos/docs-content/solutions
MAX_FILES: 50
Chunks ready: 295 (from up to 50 .md files)


## 4. Create index + bulk ingest through ELSER pipeline

| Field | Type | Role |
|-------|------|------|
| `content` | `text` | Chunk text + **keyword** search |
| `content_embedding` | `sparse_vector` | **ELSER token weights** at ingest |
| `source` | `keyword` | Citation path |

Expect **several minutes** for hundreds of chunks (ELSER runs per document).

In [19]:
from elasticsearch.helpers import bulk
from tqdm.auto import tqdm

INDEX_MAPPING = {
    "mappings": {
        "properties": {
            "content_embedding": {"type": "sparse_vector"},
            "content": {"type": "text"},
            "source": {"type": "keyword"},
        }
    }
}


def es_doc_id(raw_id: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", raw_id)[:512]


if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print("Deleted existing index:", INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
print("Created index:", INDEX_NAME)


def generate_actions():
    for d in documents:
        yield {
            "_index": INDEX_NAME,
            "_id": es_doc_id(d["id"]),
            "_source": {
                "content": d["document"],
                "source": d["metadata"]["source"],
            },
        }


print("Bulk indexing through ELSER pipeline — this can take a while...")
success, errors = bulk(
    es,
    generate_actions(),
    chunk_size=20,
    pipeline=PIPELINE_ID,
    raise_on_error=False,
    request_timeout=300,
)
print(f"Indexed {success} chunks into {INDEX_NAME}")
if errors:
    print(f"Bulk errors: {len(errors)} — first error:")
    print(errors[0])

es.indices.refresh(index=INDEX_NAME)
print("Index refreshed — ready to search.")

Deleted existing index: elastic-rag-solutions-elser
Created index: elastic-rag-solutions-elser
Bulk indexing through ELSER pipeline — this can take a while...


/var/folders/7p/_yypd4_15nq1c_w_fbmhw8c80000gn/T/ipykernel_38923/3821493500.py:40: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  success, errors = bulk(


Indexed 295 chunks into elastic-rag-solutions-elser
Index refreshed — ready to search.


## 5. Retrieve with hybrid search (keyword + sparse ELSER)

Same RRF pattern as the dense Elastic notebook, but the semantic leg uses **`sparse_vector`** + `inference_id` instead of `semantic_text`.

In [21]:
QUESTION = "What are the three main Elastic use cases and what should I use for each?"

TOP_K = 5

search_body = {
    "size": TOP_K,
    "retriever": {
        "rrf": {
            "retrievers": [
                {
                    "standard": {
                        "query": {"match": {"content": QUESTION}}
                    }
                },
                {
                    "standard": {
                        "query": {
                            "sparse_vector": {
                                "field": "content_embedding",
                                "inference_id": ELSER_INFERENCE_ID,
                                "query": QUESTION,
                            }
                        }
                    }
                },
            ]
        }
    },
}

response = es.search(index=INDEX_NAME, body=search_body)
hits = response["hits"]["hits"]

print("QUESTION:", QUESTION)
print("\n--- TOP CHUNKS (read these first) ---\n")
for rank, hit in enumerate(hits, start=1):
    src = hit["_source"]
    score = hit.get("_score", 0)
    doc = src.get("content", "")
    path = src.get("source", "?")
    print(f"### {rank}. {path}  (score: {score:.4f})")
    print(doc[:900] + ("…" if len(doc) > 900 else ""))
    print()

retrieved_docs = [h["_source"]["content"] for h in hits]
retrieved_sources = [h["_source"].get("source", "?") for h in hits]

QUESTION: What are the three main Elastic use cases and what should I use for each?

--- TOP CHUNKS (read these first) ---

### 1. index.md  (score: 0.0328)
# Solutions and use cases

:::{tip}
New to Elastic? Refer to [Elastic Fundamentals](/get-started/index.md) to understand the {{stack}}, its components, and your deployment options.
:::

Elastic helps you build applications for three main use cases: search, observability, and security. You can work directly with platform capabilities through APIs, use pre-built solutions with integrated UIs, or combine both approaches.

## Choose your path

| Your use case | What to use | Description |
| --- | --- | --- |
| Building search-powered applications | 1. [Core search capabilities](/solutions/search.md)<br><br> 2. [Elasticsearch solution](/solutions/elasticsearch-solution-project.md) | 1. Core {{es}} search features available across all deployment types, solutions, and project types<br><br>2. Additional UI tools that complement the core se

## 6. Build the RAG prompt

Unchanged — LLM sees text only.

In [22]:
def build_rag_prompt(question: str, retrieved_docs: list[str], sources: list[str]) -> str:
    blocks = []
    for i, (text, src) in enumerate(zip(retrieved_docs, sources), start=1):
        blocks.append(f"[Source {i}: {src}]\n{text}")
    context = "\n\n".join(blocks)
    return (
        "You are assisting a reader of Elastic documentation. "
        "Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. "
        "After each factual claim, add [Source N] matching the source blocks.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    )


prompt = build_rag_prompt(QUESTION, retrieved_docs, retrieved_sources)
print("--- BEGIN PROMPT ---")
print(prompt[:4000] + ("\n…" if len(prompt) > 4000 else ""))
print("--- END PROMPT (truncated if long) ---")

--- BEGIN PROMPT ---
You are assisting a reader of Elastic documentation. Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. After each factual claim, add [Source N] matching the source blocks.

CONTEXT:
[Source 1: index.md]
# Solutions and use cases

:::{tip}
New to Elastic? Refer to [Elastic Fundamentals](/get-started/index.md) to understand the {{stack}}, its components, and your deployment options.
:::

Elastic helps you build applications for three main use cases: search, observability, and security. You can work directly with platform capabilities through APIs, use pre-built solutions with integrated UIs, or combine both approaches.

## Choose your path

| Your use case | What to use | Description |
| --- | --- | --- |
| Building search-powered applications | 1. [Core search capabilities](/solutions/search.md)<br><br> 2. [Elasticsearch solution](/solutions/elasticsearch-solution-project.md) | 1. Core {{es}} search features available across a

## 7. (Optional) Ollama answer

In [23]:
import json
import urllib.error
import urllib.request

OLLAMA_MODEL = "llama3.2"
payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data.get("response", data))
except urllib.error.URLError as e:
    print("Ollama not reachable:", e)

Based on the provided context, there are three main Elastic use cases:

1. **Building search-powered applications**: You can use either Core search capabilities available across all deployment types, solutions, and project types or Elasticsearch solution with integrated UIs.
2. **Monitoring applications or infrastructure**: Use the Observability solution to monitor and troubleshoot with logs, metrics, and traces.
3. **Protecting against threats**: Use the Security solution to detect and respond to security threats.

For dynamically adjusting search fields, you should refer to [Source 4: elasticsearch-solution-project/search-applications/search-application-client.md] as it provides information on request schema for different types of queries, including combinations of text search, kNN search, ELSER search, hybrid search with RRF, and more.


## 8. Compare all three notebooks

Same `QUESTION`, same `CORPUS_FOLDER` (and similar `MAX_FILES`):

| Notebook | Semantic type | Retrieve |
|----------|---------------|----------|
| Chroma | Dense (MiniLM) | Vector only |
| Elastic dense | Dense (`semantic_text`) | Hybrid RRF |
| **This notebook** | **Sparse (ELSER)** | **Hybrid RRF** |

**Cleanup:** delete index `elastic-rag-solutions-elser` in Kibana when done (or re-run §4).